In [38]:
import pandas as pd
import glob, os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 700)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans


# 아래는 제 컴퓨터에서 압축 파일을 풀어 놓은 디렉터리이니, 여러분의 디렉터리를 설정해 주세요  
path = r'C:\big21\SEMI-PROJECT\topics'
# path로 지정한 디렉터리 밑에 있는 모든 .data 파일들의 파일명을 리스트로 취합
all_files = glob.glob(os.path.join(path, "*.data"))    
filename_list = []
opinion_text = []


# 개별 파일들의 파일명은 filename_list 리스트로 취합,
# 개별 파일들의 파일 내용은 DataFrame 로딩 후 다시 string으로 변환하여 opinion_text 리스트로 취합
for file_ in all_files:
    # 개별 파일을 읽어서 DataFrame으로 생성
    df = pd.read_table(file_,index_col=None, header=0,encoding='latin1')
   
    # 절대경로로 주어진 file 명을 가공. 만일 Linux에서 수행시에는 아래 \\를 / 변경.
    # 맨 마지막 .data 확장자도 제거
    filename_ = file_.split('\\')[-1]
    filename = filename_.split('.')[0]


    # 파일명 리스트와 파일 내용 리스트에 파일명과 파일 내용을 추가.
    filename_list.append(filename)
    opinion_text.append(df.to_string())


# 파일명 리스트와 파일 내용 리스트를  DataFrame으로 생성
document_df = pd.DataFrame({'filename':filename_list, 'opinion_text':opinion_text})
document_df.head()


,filename,opinion_text
0,accuracy_garmin_nuvi_255W_gps,", and is very, very accurate .\n0 but for the most part, we find that the Garmin software provides accurate directions, whereever we intend to go .\n1 This functi..."
1,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ..."
2,battery-life_amazon_kindle,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ..."
3,battery-life_ipod_nano_8gb,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...
4,battery-life_netbook_1005ha,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now..."


In [28]:
from nltk.stem import WordNetLemmatizer
import nltk
import string


remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
lemmar = WordNetLemmatizer()


# 입력으로 들어온 token단어들에 대해서 lemmatization 어근 변환.
def LemTokens(tokens):
    return [lemmar.lemmatize(token) for token in tokens]


# TfidfVectorizer 객체 생성 시 tokenizer인자로 해당 함수를 설정하여 lemmatization 적용
# 입력으로 문장을 받아서 stop words 제거-> 소문자 변환 -> 단어 토큰화 -> lemmatization 어근 변환.
def LemNormalize(text):
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english' , \
                             ngram_range=(1,2), min_df=0.05, max_df=0.85 )


#opinion_text 컬럼값으로 feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans  # 1. KMeans 라이브러리 추가

tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english' , \
                             ngram_range=(1,2), min_df=0.05, max_df=0.85 )

# opinion_text 컬럼값으로 feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

# 2. 군집화 모델 생성 및 학습 (예: 5개 군집)
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)

# 3. 학습된 모델에서 레이블(cluster_label) 추출 ◀ 이 부분이 누락되어 있었습니다!
cluster_label = km_cluster.labels_

# 데이터프레임에 군집 레이블 추가
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,accuracy_garmin_nuvi_255W_gps,", and is very, very accurate .\n0 but for the most part, we find that the Garmin software provides accurate directions, whereever we intend to go .\n1 This functi...",2
1,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",1
2,battery-life_amazon_kindle,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ...",4
3,battery-life_ipod_nano_8gb,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,4
4,battery-life_netbook_1005ha,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now...",4


In [39]:
import string
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# 문장 부호 제거 사전 및 Lemmatizer 정의
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
lemmar = WordNetLemmatizer()

def LemTokens(tokens):
    return [lemmar.lemmatize(token) for token in tokens]

# [★수정] to_string()으로 생성된 행 인덱스 숫자 노이즈를 필터링하는 안전장치 추가
def LemNormalize(text):
    if not isinstance(text, str):
        return []
    
    # 1. 소문자 변환 및 문장부호 제거
    clean_text = text.lower().translate(remove_punct_dict)
    
    # 2. 토큰화 진행
    tokens = nltk.word_tokenize(clean_text)
    
    # 3. [핵심] to_string() 때문에 생긴 '순수 숫자(인덱스 번호)' 토큰은 
    # 분석에서 제외하여 empty vocabulary 에러를 차단합니다.
    valid_tokens = [token for token in tokens if not token.isdigit()]
    
    return LemTokens(valid_tokens)

# --- 기존에 작성하신 코드가 그대로 흘러갑니다 ---

# TF-IDF 객체 생성 (기존 조건 그대로 유지)
tfidf_vect = TfidfVectorizer(
    tokenizer=LemNormalize, 
    stop_words='english',
    ngram_range=(1, 2), 
    min_df=0.05, 
    max_df=0.85
)

# opinion_text 컬럼값(to_string() 결과물)으로 feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])
print("🎉 [to_string 유지] TF-IDF 행렬 변환 완료! 행렬 크기:", feature_vect.shape)

# 군집화 모델 생성 및 학습 (5개 군집)
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)

# 레이블 추출 및 데이터프레임 할당
cluster_label = km_cluster.labels_
document_df['cluster_label'] = cluster_label

print("\n--- 군집별 문서 수 분포 ---")
print(document_df['cluster_label'].value_counts())

🎉 [to_string 유지] TF-IDF 행렬 변환 완료! 행렬 크기: (51, 4307)

--- 군집별 문서 수 분포 ---
cluster_label
1    15
2    13
3     9
4     8
0     6
Name: count, dtype: int64


In [40]:
# 1. 군집별로 어떤 파일(filename)들이 묶였는지 리스트 확인하기각 군집에 어떤 문서들이 들어갔는지 파일명을 정렬해서 출력하는 코드입니다. \
# 파일 이름만 봐도 "아, 이 군집은 네비게이션에 대한 거구나", "이 군집은 호텔에 대한 거구나" 하고 금방 감을 잡으실 수 있습니다.
# 군집별로 파일명을 정렬하여 확인
for i in range(5):
    print(f"\n================ [Cluster #{i}] 문서 목록 ================")
    # 해당 군집 라벨을 가진 파일명만 추출
    cluster_files = document_df[document_df['cluster_label'] == i]['filename'].values
    print(cluster_files)


================ [Cluster #0] 문서 목록 ================
<StringArray>
[        'buttons_amazon_kindle', 'eyesight-issues_amazon_kindle',
           'fonts_amazon_kindle',      'navigation_amazon_kindle',
           'price_amazon_kindle',      'price_holiday_inn_london']
Length: 6, dtype: str

================ [Cluster #1] 문서 목록 ================
<StringArray>
[ 'bathroom_bestwestern_hotel_sfo',         'food_holiday_inn_london',
          'food_swissotel_chicago',      'free_bestwestern_hotel_sfo',
  'location_bestwestern_hotel_sfo',     'location_holiday_inn_london',
   'parking_bestwestern_hotel_sfo',     'rooms_bestwestern_hotel_sfo',
         'rooms_swissotel_chicago',         'room_holiday_inn_london',
   'service_bestwestern_hotel_sfo',      'service_holiday_inn_london',
 'service_swissotel_hotel_chicago',     'staff_bestwestern_hotel_sfo',
         'staff_swissotel_chicago']
Length: 15, dtype: str

================ [Cluster #2] 문서 목록 ================
<StringArray>
[  'accuracy_garm

In [41]:
# 2. 각 군집의 중심점(Centroid)을 이용한 핵심 키워드 Top 10 추출하기KMeans 모델의 cluster_centers_
#  배열을 활용하면, 각 군집을 대표하는 가장 중요한 단어 10개를 뽑아낼 수 있습니다. 
# 이 키워드들을 보면 군집의 성격을 더 명확하게 파악할 수 있습니다.
import numpy as np

# KMeans 중심점 위치 정보 가져오기
cluster_centers = km_cluster.cluster_centers_

# TF-IDF에 사용된 단어 이름(Feature 이름) 리스트 가져오기
feature_names = tfidf_vect.get_feature_names_out()

# 군집별 핵심 단어를 추출하는 함수
def get_cluster_details(cluster_model, cluster_data, feature_names, clusters_num, top_n_features=10):
    cluster_details = {}
    
    # 각 군집의 중심점 벡터를 정렬하여 값이 큰 순서대로 인덱스를 반환
    # argsort()[:, ::-1]은 큰 값부터 거꾸로 정렬하는 것을 의미합니다.
    centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:, ::-1]
    
    for cluster_num in range(clusters_num):
        cluster_details[cluster_num] = {}
        cluster_details[cluster_num]['cluster'] = cluster_num
        
        # 상위 top_n_features(10개) 만큼의 단어 인덱스를 추출
        top_feature_indexes = centroid_feature_ordered_ind[cluster_num, :top_n_features]
        # 인덱스를 실제 단어로 변환
        top_features = [feature_names[ind] for ind in top_feature_indexes]
        
        # 해당 군집의 중심점 좌표값(중요도 점수)도 함께 추출
        top_feature_values = cluster_model.cluster_centers_[cluster_num, top_feature_indexes]
        
        cluster_details[cluster_num]['top_features'] = top_features
        cluster_details[cluster_num]['top_features_value'] = top_feature_values
        
    return cluster_details

# 함수 실행하여 상세 정보 얻기
cluster_details = get_cluster_details(km_cluster, document_df, feature_names, clusters_num=5, top_n_features=10)

# 결과 예쁘게 화면에 출력하기
for cluster_num, detail in cluster_details.items():
    print(f"\n======= [Cluster #{cluster_num}] 핵심 키워드 =======")
    print(detail['top_features'])


======= [Cluster #0] 핵심 키워드 =======
['price', 'kindle', 'page', 'button', 'font', 'book', 'eye', 'navigation', 'font size', 'easy']

======= [Cluster #1] 핵심 키워드 =======
['room', 'hotel', 'service', 'staff', 'food', 'location', 'bathroom', 'clean', 'parking', 'room service']

======= [Cluster #2] 핵심 키워드 =======
['screen', 'keyboard', 'direction', 'map', 'voice', 'feature', 'speed limit', 'speed', 'accurate', 'satellite']

======= [Cluster #3] 핵심 키워드 =======
['interior', 'seat', 'mileage', 'comfortable', 'gas', 'gas mileage', 'transmission', 'quality', 'car', 'ride']

======= [Cluster #4] 핵심 키워드 =======
['battery', 'performance', 'battery life', 'life', 'video', 'sound', 'faster', 'ipod', 'sound quality', 'camera']


In [46]:
# 1. 중심점(cluster_centers_) 객체의 형태(Shape) 출력
# 질문하신 주석의 (3, 4610)처럼 (군집수, 단어수) 구조가 출력됩니다.
# 현재 모델 기준으로는 (5, 4307)이 출력될 것입니다.
print("=== cluster_centers_의 실제 Shape ===")
print(km_cluster.cluster_centers_.shape)
print("=" * 50)

# 2. 실제 내부의 수학적 TF-IDF 중심점 값 배열(NumPy Array) 출력
# 모든 값을 출력하면 화면이 너무 길어지므로 상위 일부만 깔끔하게 출력합니다.
print("=== 실제 중심점 벡터 값 (상위 일부) ===")
print(km_cluster.cluster_centers_)

=== cluster_centers_의 실제 Shape ===
(5, 4307)
=== 실제 중심점 벡터 값 (상위 일부) ===
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.00195345 0.00153984]
 [0.0027286  0.04669153 0.00412187 ... 0.01360261 0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.0064889  0.         0.         ... 0.         0.         0.        ]]


In [47]:
import pandas as pd

# TF-IDF에 사용된 4,307개의 단어 이름 리스트 가져오기
feature_names = tfidf_vect.get_feature_names_out()

# 중심점 배열을 데이터프레임으로 변환
# 행 이름(index)은 Cluster 0~4, 열 이름(columns)은 실제 단어명
centers_df = pd.DataFrame(
    data=km_cluster.cluster_centers_, 
    index=[f'Cluster {i}' for i in range(5)], 
    columns=feature_names
)

# 데이터프레임의 앞부분 확인 (일부 단어들의 군집별 점수 매칭 상태)
print("=== 단어 이름과 매칭된 군집 중심점 데이터프레임 ===")
centers_df.iloc[:, :10] # 앞쪽 10개 단어만 샘플로 확인


=== 단어 이름과 매칭된 군집 중심점 데이터프레임 ===


,1005ha,255w,255w doe,255w scratchiness,2gb,2nd,2nd gen,30gb,30gb version,3rd
Cluster 0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Cluster 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.001107,0.000000,0.000000,0.000000,0.001377
Cluster 2,0.002729,0.046692,0.004122,0.003584,0.002834,0.002904,0.001879,0.001879,0.001879,0.000000
Cluster 3,0.000000,0.000000,0.000000,0.000000,0.000000,0.001223,0.000000,0.000000,0.000000,0.003226
Cluster 4,0.006489,0.000000,0.000000,0.000000,0.009014,0.006939,0.008365,0.002554,0.002554,0.002057


In [31]:
# NLTK 리소스 다운로드 (최초 1회 필수)
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')


def clean_and_tag_opinion(text):
    """
    to_string()으로 변환되면서 생긴 인덱스 및 노이즈를 제거하고
    단어/품사(Word/POS) 형태로 전처리하는 함수
    """
    # 1. df.to_string()으로 인해 생긴 행 번호(숫자 열) 및 불필요한 줄바꿈 공백 정제
    # 문장 시작 부분의 숫자와 공백 제거
    clean_lines = []
    for line in text.split('\n'):
        line = line.strip()
        # 판다스 to_string 결과물 특성상 앞에 붙는 인덱스 번호 제거
        line = re.sub(r'^\d+\s+', '', line) 
        if line and not line.startswith('Empty DataFrame'): # 예외 처리
            clean_lines.append(line.lower()) # 소문자 변환
            
    # 2. 정제된 문장들을 돌며 토큰화 및 품사 태깅 수행
    processed_sentences = []
    for line in clean_lines:
        # 특수문자 부호 간단히 정제 (단, 마침표 등은 유지)
        line = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?]', '', line)
        
        # 토큰화
        tokens = word_tokenize(line)
        # 품사 태깅
        pos_tags = nltk.pos_tag(tokens)
        
        # 오피노시스 포맷인 "단어/품사" 형태로 결합
        word_pos_list = [f"{word}/{tag}" for word, tag in pos_tags]
        
        # 공백으로 연결하여 하나의 전처리된 문장 문자열로 생성
        processed_sentences.append(" ".join(word_pos_list))
        
    # 모든 문장을 줄바꿈으로 합쳐서 반환
    return "\n".join(processed_sentences)

# --- 기존 document_df에 전처리 함수 적용 ---
# 새 컬럼 'cleaned_opinion'에 전처리된 텍스트 저장
document_df['cleaned_opinion'] = document_df['opinion_text'].apply(clean_and_tag_opinion)

# 결과 확인
document_df[['filename', 'cleaned_opinion']].head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mega\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mega\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


,filename,cleaned_opinion
0,accuracy_garmin_nuvi_255W_gps,",/, and/CC is/VBZ very/RB ,/, very/RB accurate/JJ ./.\nbut/CC for/IN the/DT most/JJS part/NN ,/, we/PRP find/VBP that/IN the/DT garmin/NN software/NN provides/VBZ accurate/JJ directions/NNS ,/, whereever/IN we/PRP intend/VBP to/TO go/VB ./.\nthis/DT function/NN is/VBZ not/RB accurate/JJ if/IN you/PRP dont/VBP leave/VB it/PRP in/IN battery/NN mode/NN say/VBP ,/, when/WRB you/PRP stop/VBP at/IN the/DT cracker/NN barrell/NN for/IN lunch/NN and/CC to/TO play/VB one/CD of/IN those/DT trangle/NNS games/NNS with/IN the/DT tees/NNS ./.\nit/PRP provides/VBZ immediate/JJ alternatives/NNS if/IN the/DT route/NN from/IN the/DT online/JJ map/NN program/NN was/VBD inaccurate/JJ or/CC blocked/VBN by/IN ..."
1,bathroom_bestwestern_hotel_sfo,"the/DT room/NN was/VBD not/RB overly/RB big/JJ ,/, but/CC clean/JJ and/CC very/RB comfortable/JJ beds/NNS ,/, a/DT great/JJ shower/NN and/CC very/RB clean/JJ bathrooms/NNS ./.\nthe/DT second/JJ room/NN was/VBD smaller/JJR ,/, with/IN a/DT very/RB inconvenient/JJ bathroom/NN layout/NN ,/, but/CC at/IN least/JJS it/PRP was/VBD quieter/RB and/CC we/PRP were/VBD able/JJ to/TO sleep/VB ./.\nlarge/JJ comfortable/JJ room/NN ,/, wonderful/JJ bathroom/NN ./.\nthe/DT rooms/NNS were/VBD nice/JJ ,/, very/RB comfy/JJ bed/NN and/CC very/RB clean/JJ bathroom/NN ./.\nbathroom/NN was/VBD spacious/JJ too/RB and/CC very/RB clean/JJ ./.\nthe/DT bathroom/NN only/RB had/VBD a/DT single/JJ sink/NN ,/, but/CC i..."
2,battery-life_amazon_kindle,"after/IN i/NN plugged/VBD it/PRP in/IN to/TO my/PRP$ usb/JJ hub/NN on/IN my/PRP$ computer/NN to/TO charge/VB the/DT battery/NN the/DT charging/NN cord/NN design/NN is/VBZ very/RB clever/JJ !/.\nafter/IN you/PRP have/VBP paged/VBN tru/VBP a/DT 500/CD ,/, page/NN book/NN one/CD ,/, page/NN ,/, at/IN ,/, a/DT ,/, time/NN to/TO get/VB from/IN chapter/NN 2/CD to/TO chapter/NN 15/CD ,/, see/VB how/WRB excited/JJ you/PRP are/VBP about/IN a/DT low/JJ battery/NN and/CC all/PDT the/DT time/NN it/PRP took/VBD to/TO get/VB there/RB !/.\nno/DT user/NN replaceable/JJ battery/NN ,/, ,/, unless/IN you/PRP buy/VBP the/DT extended/JJ warranty/NN for/IN 65/CD ./.\nafter/IN 1/CD year/NN you/PRP pay/VBP 80/C..."
3,battery-life_ipod_nano_8gb,"short/JJ battery/NN life/NN i/NN moved/VBD up/RB from/IN an/DT 8gb/CD ./.\ni/RB love/VBP this/DT ipod/NN except/IN for/IN the/DT battery/NN life/NN ./.\nlong/RB battery/NN scratch/NN resistant/NN\nbattery/NN drains/NNS even/RB if/IN i/JJ dont/VBP use/NN it/PRP ./.\ni/NN only/RB wonder/VB why/WRB the/DT battery/NN seems/VBZ to/TO drain/VB when/WRB im/NN not/RB using/VBG it/PRP ,/, even/RB after/IN sliding/VBG the/DT top/JJ control/NN button/NN to/TO off/RP when/WRB shutting/VBG down/RP ./.\ngreat/JJ in/IN the/DT car/NN ,/, light/NN ,/, portable/JJ ,/, good/JJ quality/NN ,/, long/JJ battery/NN ,/, scratch/NN resistant/NN ./.\n5g/CD lies/VBZ a/DT more/RBR mature/JJ ipod/NN ,/, many/JJ steps..."
4,battery-life_netbook_1005ha,"6ghz/CD 533fsb/CD cpu/NN ,/, glossy/JJ display/NN ,/, 3/CD ,/, cell/NN 23wh/CD li/NN ,/, ion/NN battery/NN ,/, and/CC a/DT 1/CD ./.\nnot/RB to/TO mention/VB that/IN as/IN of/IN now/RB asus/NN will/MD not/RB sell/VB you/PRP a/DT spare/JJ 3/CD or/CC 6/CD ,/, cell/NN li/NN ,/, ion/NN battery/NN ./.\nit/PRP also/RB features/VBZ a/DT n270/JJ cpu/NN ,/, 6/CD ,/, cell/NN 48wh/CD li/NN ,/, ion/NN battery/NN 8/CD ./.\n3mp/CD webcam/NN ,/, 6/CD ,/, cell/NN 63wh/CD li/NN ,/, ion/NN battery/NN with/IN a/DT whopping/JJ 10/CD ./.\nrealistic/JJ battery/NN numbers/NNS are/VBP between/IN 8/CD ./.\nof/IN battery/NN life/NN if/IN youre/NN using/VBG wifi/JJ doing/VBG email/JJ word/NN processing/NN youtube/N..."


In [32]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# 1. Lemmatizer 초기화
lemmar = WordNetLemmatizer()

# 2. 텍스트를 받아서 "이미 완벽하게 토큰화된 단어 리스트"를 반환하는 함수
def LemNormalize_Final_List(text):
    if not isinstance(text, str):
        return []
    
    # '단어/품사' 구조에서 순수 문자 및 숫자 부분만 결합하여 추출
    word_pos_pairs = re.findall(r'([a-zA-Z0-9_]+)/([a-zA-Z\$,\.]+)', text)
    
    lemmed_tokens = []
    for word, pos in word_pos_pairs:
        word = word.lower()
        
        # 순수한 기호나 불필요한 공백 제거
        if not word.isalnum():
            continue
            
        # 품사 정보를 기반으로 어근 변환
        if pos.startswith('VB'):    # 동사
            word = lemmar.lemmatize(word, pos='v')
        elif pos.startswith('JJ'):  # 형용사
            word = lemmar.lemmatize(word, pos='a')
        elif pos.startswith('NN'):  # 명사
            word = lemmar.lemmatize(word, pos='n')
        else:
            word = lemmar.lemmatize(word)
            
        lemmed_tokens.append(word)
        
    return lemmed_tokens

# 3. TF-IDF 벡터화 수행 (핵심 피처 제한 우회 설정)
# token_pattern=None과 lowercase=False를 주어 사이킷런 내부의 자동 단어 삭제 기능을 끕니다.
tfidf_vect = TfidfVectorizer(
    tokenizer=LemNormalize_Final_List,
    token_pattern=None,      # ◀ 내부 엔진의 2글자 미만 단어 강제 삭제 기능을 끕니다.
    lowercase=False,         # ◀ 이미 우리가 소문자 변환을 했으므로 이중 변환 노이즈를 끕니다.
    stop_words='english',    # 기본 불용어 제외
    ngram_range=(1, 2), 
    min_df=1,                # ◀ 빈 사전을 방지하기 위해 최소 빈도를 1로 고정합니다.
    max_df=0.85
)

# feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])
print("🎉 [성공] TF-IDF 행렬 변환 완료! 행렬 크기:", feature_vect.shape)

# 4. K-Means 군집화 모델 생성 및 학습 (5개 군집)
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)

# 5. 결과를 데이터프레임에 안전하게 저장
document_df['cluster_label'] = km_cluster.labels_

print("\n--- 군집별 문서 분포 결과 ---")
print(document_df['cluster_label'].value_counts())

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [24]:
def clean_for_opinosis_graph(text):
    """
    이미 태깅된 텍스트의 줄바꿈(\n)과 문장 부호 태그들을 깔끔하게 정리하는 함수
    """
    # 개별 문장 단위로 분리
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # 문장 부호 태그 제거 원할 경우 (선택 사항)
        # line = re.sub(r'[\.\,\!\?\/]+[\.\,\!\?\/]+', '', line)
        
        cleaned_lines.append(line)
        
    return "\n".join(cleaned_lines)

# 오피노시스 입력용 컬럼 생성
document_df['opinosis_input'] = document_df['opinion_text'].apply(clean_for_clustering)

In [21]:
# [체크용] 데이터가 공백이 아니라 잘 변환되었는지 임의로 출력해보기
print("--- 전처리 결과 샘플 ---")
print(document_df['cluster_text'].iloc[0][:200]) 

# 3. TF-IDF 벡터화 (데이터 개수가 적을 때를 대비해 min_df 조건 완화)
# 문서 수가 적으면 min_df=0.05 조건 때문에 단어가 다 잘려 나갈 수 있으므로 1~2개 문서 이상 등장으로 변경
tfidf_vect = TfidfVectorizer(
    stop_words='english', 
    ngram_range=(1, 2), 
    max_df=0.85, 
    min_df=1 # 최소 1개 혹은 2개 이상의 문서에 등장하는 단어는 모두 포함
)


--- 전처리 결과 샘플 ---



In [22]:
feature_vect = tfidf_vect.fit_transform(document_df['cluster_text'])

# 4. K-Means 군집화 진행
NUM_CLUSTERS = 3 
km_cluster = KMeans(n_clusters=NUM_CLUSTERS, max_iter=10000, random_state=42)
km_cluster.fit(feature_vect)

document_df['cluster_label'] = km_cluster.labels_

print("\n--- 군집화 성공! 빈도수 결과 ---")
print(document_df['cluster_label'].value_counts())

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [8]:
import networkx as nx
import matplotlib.pyplot as plt

def build_opinosis_graph(text_data):
    # 인접한 단어들의 연결 빈도를 기록하기 위한 유향 그래프(Directed Graph) 생성
    G = nx.DiGraph()
    
    # 줄바꿈(\n) 단위로 개별 문장 분리
    sentences = text_data.split('\n')
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        # 문장 안의 "단어/품사" 토큰들을 공백 기준으로 분리
        tokens = sentence.split()
        
        # 문장 내 단어들을 순서대로 링크 연결
        for i in range(len(tokens) - 1):
            node_v1 = tokens[i]      # 현재 단어 (예: 'very/RB')
            node_v2 = tokens[i+1]    # 다음 단어 (예: 'accurate/JJ')
            
            # 그래프에 이미 해당 연결(Edge)이 있다면 가중치(weight) + 1
            if G.has_edge(node_v1, node_v2):
                G[node_v1][node_v2]['weight'] += 1
            else:
                # 처음 발견된 연결이라면 가중치 1로 새로 생성
                G.add_edge(node_v1, node_v2, weight=1)
                
    return G

# --- 실행 및 시각화 예시 ---
# document_df에서 첫 번째 파일(예: 가민 GPS 정확도 리뷰)의 텍스트를 가져왔다고 가정합니다.
sample_text = """
,/, and/CC is/VBZ very/RB ,/, very/RB accurate/JJ ./.
but/CC for/IN the/DT most/JJS part/NN ,/, we/PRP find/VBP that/IN the/DT garmin/NN software/NN provides/VBZ accurate/JJ directions/NNS
this/DT function/NN is/VBZ not/RB accurate/JJ if/IN you/PRP dont/VBP leave/VB it/PRP
"""

# 1. 그래프 빌드
opinosis_graph = build_opinosis_graph(sample_text)

# 2. 결과 확인 (가장 많이 중복 연결된 상위 5개 경로 출력)
print("--- 빈도가 높은 연결 경로 (Edge) ---")
edges_sorted = sorted(opinosis_graph.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
for u, v, weight in edges_sorted[:5]:
    print(f"[{u}] ➔ [{v}] (빈도: {weight['weight']})")

--- 빈도가 높은 연결 경로 (Edge) ---
[,/,] ➔ [and/CC] (빈도: 1)
[,/,] ➔ [very/RB] (빈도: 1)
[,/,] ➔ [we/PRP] (빈도: 1)
[and/CC] ➔ [is/VBZ] (빈도: 1)
[is/VBZ] ➔ [very/RB] (빈도: 1)


In [37]:
import string
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# 최초 1회 필요한 NLTK 데이터 다운로드
nltk.download('punkt')
nltk.download('wordnet')

# 문장 부호 제거 사전 정의
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
lemmar = WordNetLemmatizer()

# 1. 입력으로 들어온 토큰단어들에 대해 Lemmatization(어근 변환) 수행
def LemTokens(tokens):
    return [lemmar.lemmatize(token) for token in tokens]

# 2. 소문자 변환 -> 문장부호 제거 -> 토큰화 -> Lemmatization 수행
def LemNormalize(text):
    if not isinstance(text, str):
        return []
    # 문장 부호를 완전히 지운 후 토큰화하여 어근을 변환합니다.
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))

# 3. TF-IDF 벡터화 객체 생성 (기존에 작성하셨던 설정을 그대로 유지합니다)
# 이제 유효한 일반 단어들이 정상적으로 처리되므로 원래 조건(min_df=0.05)이 아주 잘 작동합니다.
tfidf_vect = TfidfVectorizer(
    tokenizer=LemNormalize, 
    stop_words='english',
    ngram_range=(1, 2), 
    min_df=0.05, 
    max_df=0.85
)

# feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])
print("🎉 [드디어 성공] TF-IDF 행렬 변환 완료! 행렬 크기:", feature_vect.shape)

# 4. K-Means 군집화 모델 생성 및 학습 (5개 군집)
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)

# 5. 학습된 모델에서 레이블 추출 후 데이터프레임에 추가
cluster_label = km_cluster.labels_
document_df['cluster_label'] = cluster_label

print("\n--- 군집별 문서 수 분포 ---")
print(document_df['cluster_label'].value_counts())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mega\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mega\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


🎉 [드디어 성공] TF-IDF 행렬 변환 완료! 행렬 크기: (51, 4412)

--- 군집별 문서 수 분포 ---
cluster_label
1    15
2    12
4     9
3     9
0     6
Name: count, dtype: int64
